# Project 3 Health Dataset: Full Visualization Notebook

This notebook includes:
- loading the `insurance.csv` dataset
- checking structure and data quality
- feature engineering for later merging
- all chart code together
- optional code to save the charts as PNG files


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('insurance.csv')
df.head()


## 1. Check structure and data quality


In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())


## 2. Summary statistics


In [ ]:
df.describe(include='all')


## 3. Feature engineering for later merge/scoring

These variables can help later if you want to combine this dataset conceptually with car insurance and loneliness data.


In [ ]:
df['is_smoker'] = (df['smoker'] == 'yes').astype(int)
df['is_male'] = (df['sex'] == 'male').astype(int)
df['has_children'] = (df['children'] > 0).astype(int)
df['log_charges'] = np.log(df['charges'])

df['age_group'] = pd.cut(
    df['age'],
    bins=[17, 25, 35, 45, 55, 65],
    labels=['18-25', '26-35', '36-45', '46-55', '56-64']
)

df['bmi_group'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, 100],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese'],
    right=False
)

df['charge_quartile'] = pd.qcut(
    df['charges'],
    4,
    labels=['Q1_low', 'Q2_midlow', 'Q3_midhigh', 'Q4_high']
)

df['high_burden'] = (df['charge_quartile'] == 'Q4_high').astype(int)

# Simple health vulnerability score
age_z = (df['age'] - df['age'].mean()) / df['age'].std()
bmi_z = (df['bmi'] - df['bmi'].mean()) / df['bmi'].std()
charges_z = (df['charges'] - df['charges'].mean()) / df['charges'].std()
children_z = (df['children'] - df['children'].mean()) / df['children'].std()

raw_score = (
    0.40 * charges_z +
    0.30 * df['is_smoker'] +
    0.15 * bmi_z +
    0.10 * age_z +
    0.05 * children_z
)

df['health_vulnerability_score'] = (
    (raw_score - raw_score.min()) /
    (raw_score.max() - raw_score.min()) * 100
).round(2)

df.head()


## 4. Chart 1: Distribution of insurance charges


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['charges'], bins=30)
plt.title('Distribution of Insurance Charges')
plt.xlabel('Charges')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 5. Chart 2: Insurance charges by smoker status


In [ ]:
plt.figure(figsize=(6, 5))
df.boxplot(column='charges', by='smoker')
plt.title('Insurance Charges by Smoker Status')
plt.suptitle('')
plt.xlabel('Smoker')
plt.ylabel('Charges')
plt.tight_layout()
plt.show()


## 6. Chart 3: BMI vs insurance charges


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['bmi'], df['charges'])
plt.title('BMI vs Insurance Charges')
plt.xlabel('BMI')
plt.ylabel('Charges')
plt.tight_layout()
plt.show()


## 7. Chart 4: Average charges by age group


In [ ]:
age_means = df.groupby('age_group', observed=False)['charges'].mean()

plt.figure(figsize=(7, 5))
plt.bar(age_means.index.astype(str), age_means.values)
plt.title('Average Charges by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.show()


## 8. Chart 5: Average charges by BMI group


In [ ]:
bmi_means = df.groupby('bmi_group', observed=False)['charges'].mean()

plt.figure(figsize=(7, 5))
plt.bar(bmi_means.index.astype(str), bmi_means.values)
plt.title('Average Charges by BMI Group')
plt.xlabel('BMI Group')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.show()


## 9. Chart 6: Average charges by region


In [ ]:
region_means = df.groupby('region')['charges'].mean().sort_values()

plt.figure(figsize=(7, 5))
plt.bar(region_means.index.astype(str), region_means.values)
plt.title('Average Charges by Region')
plt.xlabel('Region')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.show()


## 10. Chart 7: Distribution of health vulnerability score


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['health_vulnerability_score'], bins=25)
plt.title('Distribution of Health Vulnerability Score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## 11. Chart 8: Health vulnerability score vs charges


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['health_vulnerability_score'], df['charges'])
plt.title('Health Vulnerability Score vs Charges')
plt.xlabel('Health Vulnerability Score')
plt.ylabel('Charges')
plt.tight_layout()
plt.show()


In [ ]:
# 1
plt.figure(figsize=(8, 5))
plt.hist(df['charges'], bins=30)
plt.title('Distribution of Insurance Charges')
plt.xlabel('Charges')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('01_charges_distribution.png', dpi=200)
plt.close()

# 2
plt.figure(figsize=(6, 5))
df.boxplot(column='charges', by='smoker')
plt.title('Insurance Charges by Smoker Status')
plt.suptitle('')
plt.xlabel('Smoker')
plt.ylabel('Charges')
plt.tight_layout()
plt.savefig('02_charges_by_smoker_boxplot.png', dpi=200)
plt.close()

# 3
plt.figure(figsize=(7, 5))
plt.scatter(df['bmi'], df['charges'])
plt.title('BMI vs Insurance Charges')
plt.xlabel('BMI')
plt.ylabel('Charges')
plt.tight_layout()
plt.savefig('03_bmi_vs_charges_scatter.png', dpi=200)
plt.close()

# 4
age_means = df.groupby('age_group', observed=False)['charges'].mean()
plt.figure(figsize=(7, 5))
plt.bar(age_means.index.astype(str), age_means.values)
plt.title('Average Charges by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.savefig('04_avg_charges_by_age_group.png', dpi=200)
plt.close()

# 5
bmi_means = df.groupby('bmi_group', observed=False)['charges'].mean()
plt.figure(figsize=(7, 5))
plt.bar(bmi_means.index.astype(str), bmi_means.values)
plt.title('Average Charges by BMI Group')
plt.xlabel('BMI Group')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.savefig('05_avg_charges_by_bmi_group.png', dpi=200)
plt.close()

# 6
region_means = df.groupby('region')['charges'].mean().sort_values()
plt.figure(figsize=(7, 5))
plt.bar(region_means.index.astype(str), region_means.values)
plt.title('Average Charges by Region')
plt.xlabel('Region')
plt.ylabel('Average Charges')
plt.tight_layout()
plt.savefig('06_avg_charges_by_region.png', dpi=200)
plt.close()

# 7
plt.figure(figsize=(8, 5))
plt.hist(df['health_vulnerability_score'], bins=25)
plt.title('Distribution of Health Vulnerability Score')
plt.xlabel('Score')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('07_health_vulnerability_score_distribution.png', dpi=200)
plt.close()

# 8
plt.figure(figsize=(7, 5))
plt.scatter(df['health_vulnerability_score'], df['charges'])
plt.title('Health Vulnerability Score vs Charges')
plt.xlabel('Health Vulnerability Score')
plt.ylabel('Charges')
plt.tight_layout()
plt.savefig('08_score_vs_charges_scatter.png', dpi=200)
plt.close()

print('All chart files saved.')


- engineered fields for future integration:
- is_smoker
- is_male
- has_children
- log_charges
- age_group
- bmi_group
- charge_quartile
- high_burden
- health_vulnerability_score
- cross-group tables you can reuse later
- a simple health vulnerability score from 0–100 so this dataset can plug into a later cross-domain score

- smoker is the strongest single risk signal
- charges works well as a financial-burden variable
- age and bmi add secondary health-risk structure
- high_burden and charge_quartile are useful if later datasets are more categorical
- health_vulnerability_score gives you one summary field that can later be combined with car-risk and loneliness-risk scores


- overall_vulnerability = health_score + car_score + loneliness_score